# Module 8: Packaging and a Small Clinic CLI

**Concepts:** packages · imports · `pyproject.toml` · CLI · CI overview

> **Safety note:** This is a **programming** exercise. It is **not** medical advice, not a diagnosis, and **not** for real patients or real medicines. We use invented drug names (for example **Alercin**) and school biology/chemistry only.

### How to use this notebook

1. Read the markdown cell (the story).
2. Run the code cell under it with **Shift+Enter**.
3. Check **What you should see**.
4. Try the exercises before opening **Solutions**.

You will design a tiny tool `cliniccalc` with dose / DNA / Punnett helpers.

## 1. Module vs package

- **Module** — one `.py` file  
- **Package** — a folder with `__init__.py` (and more modules)

**What you should see:** a printed layout tree.

In [ ]:
layout = '''
cliniccalc/
  __init__.py
  dose.py
  dna.py
  genetics.py
  cli.py
pyproject.toml
'''
print(layout)

## 2. Minimal `pyproject.toml` (box label)

Think of it as the **label on the medicine box**: name, version, entry point.

**What you should see:** sample TOML text.

In [ ]:
PYPROJECT = '''
[project]
name = "cliniccalc"
version = "0.1.0"
description = "Practice clinic calculator for learning Python"
requires-python = ">=3.10"
dependencies = []

[project.scripts]
cliniccalc = "cliniccalc.cli:main"
'''
print(PYPROJECT)

## 3. Pure functions to put in the package

These are the same ideas from Module 2 — keep them free of `input()` so tests stay easy.

**What you should see:** sample outputs.

In [ ]:
def dose_mg(weight_kg, mg_per_kg=10, max_mg=1000):
    needed = weight_kg * mg_per_kg
    if needed > max_mg:
        return "OVER_MAX"
    return needed


PAIRS = {"A": "T", "T": "A", "C": "G", "G": "C"}


def complement(dna):
    return "".join(PAIRS[b] for b in dna.upper())


def punnett_counts(p1, p2):
    counts = {}
    for a in p1:
        for b in p2:
            geno = "".join(sorted([a, b], key=lambda ch: (ch.islower(), ch)))
            if geno == "aA":
                geno = "Aa"
            counts[geno] = counts.get(geno, 0) + 1
    return counts


print(dose_mg(52.5), complement("ATGC"), punnett_counts("Aa", "Aa"))

## 4. Simple menu with `input()` (works without Click)

In Jupyter, `input()` waits for you. For a non-interactive demo we call the handlers directly.

**What you should see:** three demo lines.

In [ ]:
def menu_dose(weight):
    return dose_mg(float(weight))


def menu_dna(seq):
    return complement(seq)


def menu_cross(p1, p2):
    return punnett_counts(p1, p2)


# Demo without typing (in a real CLI you would use input())
print("dose", menu_dose(52.5))
print("dna", menu_dna("ATGC"))
print("cross", menu_cross("Aa", "Aa"))

## 5. Click sketch (optional library)

If Click is missing, this cell still explains the idea.

**What you should see:** either a Click command help text or a note that Click is not installed.

In [ ]:
try:
    import click

    @click.group()
    def cli():
        # Practice clinic calculator.
        pass

    @cli.command()
    @click.option("--weight", type=float, required=True)
    def dose(weight):
        click.echo(dose_mg(weight))

    print("Click is available. Example: cliniccalc dose --weight 52.5")
except ImportError:
    print("Click not installed — the input() menu above is enough for this course.")
    print("Sketch:")
    print("  @click.command()")
    print("  @click.option('--weight', type=float)")
    print("  def dose(weight): ...")

## 6. Entry points

After `pip install -e .`, the shell command `cliniccalc` would call `cliniccalc.cli:main`.

**What you should see:** the mapping printed.

In [ ]:
print("entry point maps: cliniccalc -> cliniccalc.cli:main")

## 7. CI sketch — three steps before share

1. lint  
2. unit tests  
3. build package  

**What you should see:** a tiny YAML sketch.

In [ ]:
CI = '''
name: ci
on: [push]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.12" }
      - run: pip install -e . pytest
      - run: pytest
      - run: python -m build
'''
print(CI)

## 8. Capstone — `wardcheck` on a CSV

Read practice CSV; print who is over the fever practice cutoff.

**What you should see:** Ana flagged.

In [ ]:
from pathlib import Path

csv_text = "name,temp_c\nAna,38.1\nBogdan,36.9\nClara,39.0\n"
path = Path("_clinic_notes")
path.mkdir(exist_ok=True)
file = path / "ward.csv"
file.write_text(csv_text, encoding="utf-8")


def wardcheck(csv_path, cutoff=37.5):
    lines = Path(csv_path).read_text(encoding="utf-8").strip().splitlines()
    header, *rows = lines
    flagged = []
    for row in rows:
        name, temp_s = row.split(",")
        temp = float(temp_s)
        if temp >= cutoff:
            flagged.append((name, temp))
    return flagged


for name, temp in wardcheck(file):
    print(f"FEVER_FLAG (practice): {name} {temp}")

---
# Exercises

**Ex 1** — List two benefits of `pip install -e .` for local practice.

In [ ]:
# TODO: print two strings

**Ex 2** — Sketch a Click option `--weight` (int/float) on a fake dose command.

In [ ]:
# TODO: comments OK

**Ex 3** — Name three CI steps before publishing your practice package.

In [ ]:
# TODO

---
# Solutions

In [ ]:
print("1. edits are visible without reinstalling the package")
print("2. imports match how a real install looks")

In [ ]:
# @cli.command()
# @click.option("--weight", type=float, required=True)
# def dose(weight):
#     click.echo(dose_mg(weight))
print("see sketch above")

In [ ]:
steps = ["lint", "unit tests (pytest)", "build wheel/sdist"]
for i, s in enumerate(steps, 1):
    print(i, s)

**Course complete.** Go back to Module 1 and rebuild the ward-row parser as part of `wardcheck`. Keep using invented drugs and school biology/chemistry only.